In [ ]:
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from datetime import datetime
from pprint import pprint

current_directory = os.getcwd()
sys.path.append(current_directory)
from cnn.utils.model import CNN_Model
from cnn.utils.utils import Npy3dDataset, transform_test_3d, transform_train_3d

print(current_directory)
device = torch.device('cuda:0')

In [ ]:
print(torch.cuda.is_available())

random_seed = 42
print(f"随机种子: {random_seed}")

def set_seed(seed=random_seed):
    """设置所有随机种子以确保可重复性"""
    # Python随机种子
    random.seed(seed)
    
    # Numpy随机种子
    np.random.seed(seed)
    
    # PyTorch随机种子
    torch.manual_seed(seed)
    
    # 如果使用GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # 多GPU情况
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    # 设置环境变量
    os.environ['PYTHONHASHSEED'] = str(seed)

# 在训练开始前调用
set_seed(random_seed)

# 模型保存
model_id = datetime.now().strftime("%m%d") # 模型名
model_save_dir = f"./model/"
os.makedirs(model_save_dir, exist_ok=True)

# 参数设置
batch_size = 128
num_epochs = 500
best_val_loss = float('inf')
best_model_state = None

# 初始化模型、损失函数和优化器
model = CNN_Model(input_channels=4).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using 2 GPUs!")
    model = nn.DataParallel(model, device_ids=[0, 1]).to(device)
else:
    print("Using a single GPU.")

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)

# 创建数据集和DataLoader
train_dataset = Npy3dDataset(csv_file=current_directory + '/data/split/train.csv', transform=transform_train_3d)
valid_dataset = Npy3dDataset(csv_file=current_directory + '/data/split/valid.csv', transform=transform_test_3d)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
train_losses, val_losses = [], []
os.makedirs(model_save_dir + f"/records/checkpoints", exist_ok=True)

# 开始训练
for epoch in range(num_epochs):
    epoch_sttime = datetime.now()
    model.train()  # 设置为训练模式
    epoch_loss = 0
    
    # 每轮下采样平衡训练数据
    for inputs, labels in train_loader:
        
        # 将数据转移到GPU
        inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()  # 清除之前的梯度

        # 前向传播
        outputs = model(inputs)
        
        # 计算损失
        loss = criterion(outputs, labels)
        loss.backward()  # 反向传播
        
        optimizer.step()  # 更新权重
        epoch_loss += loss.item()  # 累积损失
    
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # 在每个epoch结束后，进行验证
    model.eval()  # 设置模型为评估模式
    val_loss = 0
    with torch.no_grad():  # 不计算梯度
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    # 计算验证集的损失
    val_loss /= len(valid_loader)
    val_losses.append(val_loss)

    # 调整学习率
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} costed: {datetime.now() - epoch_sttime} - Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
    
    # 更新模型
    if val_loss < best_val_loss:
        best_model_epoch = epoch
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        torch.save(best_model_state, model_save_dir + f"/records/checkpoints/cnn_model_{epoch}.pth")

# 保存最好的模型
torch.save(best_model_state, model_save_dir + f"/cnn_model.pth")


In [ ]:
# 保存模型
# torch.save(best_model_state, model_save_dir + f"cnn_model_{model_id}.pth")

# 绘制训练和验证损失的曲线图
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss", color="blue")
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss", color="red")
# 标注最佳模型所在 epoch（训练循环中 epoch 从 0 开始，图中横轴从 1 开始）
plt.axvline(x=best_model_epoch + 1, color="green", linestyle="--", alpha=0.8, \
            label=f"Best model (epoch {best_model_epoch + 1})")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(model_save_dir + f"/loss_hist.png")
plt.show()

# 记录 loss 并保存
import pandas as pd
loss_df = pd.DataFrame({
    'train_loss': train_losses,
    'val_loss': val_losses
})

# 保存为CSV
loss_df.to_csv(model_save_dir + f"/loss_hist.csv", index=False)


In [ ]:
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

def calculate_tss(confusion_mat):
    """计算TSS（True Skill Statistic），若只有一类样本则返回 0"""
    flat = confusion_mat.ravel()
    if len(flat) != 4:
        return 0.0
    tn, fp, fn, tp = flat
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return tpr - fpr

def save_predictions_and_metrics(model, device, save_dir):
    """保存所有数据集的预测结果和评估指标"""
    os.makedirs(save_dir, exist_ok=True)
    
    # 定义数据集列表
    datasets = {
        'train': current_directory + '/data/split/train.csv',
        'valid': current_directory + '/data/split/valid.csv', 
        'valid_1': current_directory + '/data/split/valid_1.csv',
        'valid_2': current_directory + '/data/split/valid_2.csv',
        'test': current_directory + '/data/split/test.csv', 
        'test_1': current_directory + '/data/split/test_1.csv',
        'test_2': current_directory + '/data/split/test_2.csv'
    }
    
    # 存储所有预测结果
    all_predictions = []
    all_metrics = {}
    
    # 设置模型为评估模式
    model.eval()
    
    # 对每个数据集进行预测
    for dataset_name, csv_file in datasets.items():
        print(f"\nProcessing {dataset_name} dataset...")
        
        # 创建数据集和加载器
        dataset = Npy3dDataset(csv_file=csv_file, transform=transform_test_3d)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        
        all_labels = []
        all_preds = []
        all_probabilities = []
        file_names = []
        
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
                outputs = model(inputs)
                probabilities = torch.sigmoid(outputs)  # 获取概率值
                preds = (probabilities > 0.5) * 1
                
                # 收集预测结果
                all_labels.extend(labels.cpu().numpy().flatten())
                all_preds.extend(preds.cpu().numpy().flatten())
                all_probabilities.extend(probabilities.cpu().numpy().flatten())
        
        # 获取文件名
        data_frame = pd.read_csv(csv_file)
        file_names = data_frame['file_name'].tolist()

        # 创建当前数据集的预测结果DataFrame
        df_predictions = pd.DataFrame({
            'file_name': file_names,
            'dataset': dataset_name,
            'label': all_labels,
            'prediction': all_preds,
            'probability': all_probabilities
        })
        
        # 添加到总预测结果中
        all_predictions.append(df_predictions)
        
        # 计算评估指标（zero_division=0 避免某类样本为 0 时告警）
        confusion_mat = confusion_matrix(all_labels, all_preds)
        metrics = {
            "dataset": dataset_name,
            "recall": recall_score(all_labels, all_preds, zero_division=0),
            "precision": precision_score(all_labels, all_preds, zero_division=0),
            "f1_score": f1_score(all_labels, all_preds, zero_division=0),
            "tss": calculate_tss(confusion_mat),
            "confusion_matrix": confusion_mat.tolist(),
            "samples_count": len(all_labels)
        }
        
        all_metrics[dataset_name] = metrics
        
        print(f"{dataset_name} Dataset Metrics:")
        pprint(metrics)
    
    # 合并所有预测结果
    combined_predictions = pd.concat(all_predictions, ignore_index=True)
    
    # 保存预测结果到CSV文件
    predictions_file = os.path.join(save_dir, 'all_predictions.csv')
    combined_predictions.to_csv(predictions_file, index=False)
    print(f"\n所有预测结果已保存到: {predictions_file}")
    
    # 保存评估指标到TXT文件
    metrics_file = os.path.join(save_dir, 'evaluation_metrics.txt')
    with open(metrics_file, 'w', encoding='utf-8') as f:
        f.write("模型评估指标报告\n")
        f.write("=" * 50 + "\n\n")
        
        for dataset_name, metrics in all_metrics.items():
            f.write(f"{dataset_name.upper()} 数据集评估结果:\n")
            f.write("-" * 30 + "\n")
            f.write(f"样本数量: {metrics['samples_count']}\n")
            f.write(f"召回率 (Recall): {metrics['recall']:.4f}\n")
            f.write(f"精确率 (Precision): {metrics['precision']:.4f}\n")
            f.write(f"F1分数: {metrics['f1_score']:.4f}\n")
            f.write(f"TSS分数: {metrics['tss']:.4f}\n")
            f.write(f"混淆矩阵:\n")
            cm = metrics['confusion_matrix']
            if len(cm) == 2 and len(cm[0]) == 2:
                f.write(f"  True Negative: {cm[0][0]}\n  False Positive: {cm[0][1]}\n")
                f.write(f"  False Negative: {cm[1][0]}\n  True Positive: {cm[1][1]}\n")
            else:
                f.write(f"  {cm}\n")
            f.write("\n")
    
    print(f"评估指标已保存到: {metrics_file}")
    
    # 同时保存为CSV格式便于分析
    metrics_df = pd.DataFrame.from_dict(all_metrics, orient='index')
    metrics_csv_file = os.path.join(save_dir, 'evaluation_metrics.csv')
    metrics_df.to_csv(metrics_csv_file, index_label='dataset')
    print(f"评估指标(CSV格式)已保存到: {metrics_csv_file}")
    
    return combined_predictions, all_metrics

# 训练结束后加载验证集上最优权重再评估
model.load_state_dict(best_model_state)
model = model.to(device)
model.eval()

combined_predictions, all_metrics = save_predictions_and_metrics(model, device, model_save_dir)

print("\n合并后的预测结果前5行:")
print(combined_predictions.head())
print("\n各数据集样本分布:")
print(combined_predictions['dataset'].value_counts())